## Analysera tidtabeller
* [#437](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/437)

In [2]:
from pathlib import Path
import requests
import pdfplumber
import pandas as pd
from tqdm.notebook import tqdm

In [5]:
BASE_URL = "https://kund.printhuset-sthlm.se/wa"

DOWNLOAD_DIR = Path("waxholm_pdf")
DOWNLOAD_DIR.mkdir(exist_ok=True)

routes = [str(i) for i in range(2, 32)] + [
    "40",
    "60a",
    "60b",
    "60d",
    "60e",
    "60f",
    "60g",
    "61",
    "65",
]

In [7]:
for route in tqdm(routes):

    url = f"{BASE_URL}/s{route}.pdf"

    filename = DOWNLOAD_DIR / f"s{route}.pdf"

    if filename.exists():
        continue

    r = requests.get(url)

    if r.status_code == 200:
        filename.write_bytes(r.content)
        print("Downloaded", filename.name)
    else:
        print("Missing", url)

  0%|          | 0/39 [00:00<?, ?it/s]

Missing https://kund.printhuset-sthlm.se/wa/s6.pdf
Missing https://kund.printhuset-sthlm.se/wa/s61.pdf
Missing https://kund.printhuset-sthlm.se/wa/s65.pdf


In [8]:
pdf = DOWNLOAD_DIR / "s3.pdf"

text = ""

with pdfplumber.open(pdf) as p:

    for page in p.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

print(text[:4000])

GÄLLER 19 JUNI 2026 – 16 AUGUSTI 2026 Uppdaterad 24 juni 2026
3A
STOCKHOLM – VAXHOLM – NORRA LAGNÖ – ÅLSTÄKET
DAG MÅNDAG–TORSDAG FREDAG
FARTYG
TURNUMMER
ANMÄRKNING
Strömkajen (Stockholm)
Slussen (Stockholm)
Nacka strand
Hasseludden
Gåshaga brygga
Vaxholm ank.
Vaxholm avg.
Grenadjärbryggan (Rindö)
Knutsholmen
Skogsö södra
Hästholmen
Skogsö udde
Norra Lagnö
Ålstäket
HELGTRAFIK ANSLUTANDE SL-TRAFIK
Vid jul, nyår, påsk, midsommar samt övriga storhelger Sök hela din resa på sl.se för att få tider för både
förekommer förändringar i trafiken. Se information anslutande SL-trafik och skärgårdstrafiken.
och sök din resa på waxholmsbolaget.se eller i appen.
NERAXAS
0303
06.28
06.45
ÖREVLIS
0300
06.45
07.00b
X
X
07.10
07.20
NEGGIV
0309
08.10
08.32
08.37b
X
X
08.47
08.55
09.10
TÅB-V
1401
07.45
08.00
08.10
ÖREVLIS
0320
08.15z
08.10z
08.50
NEGGIV
0309
09.32z
09.29z
09.26z
09.25z
08.55z
09.10
TÅB-V
1405
13.00
13.17
13.27
13.32
14.00
NEGGIV
0315
14.10
14.30
14.35b
X
X
14.45
14.55
15.10
TÅB-V
1305
17.00

In [9]:
documents = {}

for pdf in DOWNLOAD_DIR.glob("*.pdf"):

    txt = ""

    with pdfplumber.open(pdf) as p:

        for page in p.pages:

            t = page.extract_text()

            if t:
                txt += t + "\n"

    documents[pdf.stem] = txt

print(len(documents), "PDF loaded")

36 PDF loaded


In [10]:
import pdfplumber

pdf = DOWNLOAD_DIR / "s3.pdf"

with pdfplumber.open(pdf) as p:

    page = p.pages[0]

    tables = page.extract_tables()

print(f"{len(tables)} tables found")

1 tables found


In [12]:
table = tables[0]

for row in table[:20]:
    print(row)

import pandas as pd

df = pd.DataFrame(table)

print(df.shape)
df.head(20)

[None, 'MÅNDAG–TORSDAG', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
['FARTYG', 'NERAXAS', 'ÖREVLIS', 'NEGGIV', 'TÅB-V', None, 'ÖREVLIS', None, 'NEGGIV', 'TÅB-V', None, 'NEGGIV', 'TÅB-V', 'NEGGIV', 'ÖREBIV', 'ÖREVLIS', 'NEGGIV', 'ÖDMRÄV', None, 'ÖREVLIS', None, 'NEGGIV', 'ÖXAV', None, 'NEGGIV', 'ÖREBIV', 'RÄKSROTS', None]
['TURNUMMER', '0303', '0300', '0309', '1401', None, '0320', None, '0309', '1405', None, '0315', '1305', '0319', '0303', '0300', '0309', '1421', None, '0320', None, '0309', '1425', None, '0315', '1225', '0925', None]
['ANMÄRKNING', '', '', '', '', None, '', None, '', '', None, '', '', '', '', '', '', '', None, '', None, '', '', None, '', '', '', None]
[None, '', '', '', '07.45', None, '', None, '', '13.00', None, '', '17.00', '', '', '', '', '07.45', None, '', None, '', '13.00', None, '', '', '16.45', None]
[None, '', '', '', '', '', '', None, '', '', '', '',

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
0,None,MÅNDAG–TORSDAG,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,FARTYG,NERAXAS,ÖREVLIS,NEGGIV,TÅB-V,None,ÖREVLIS,None,NEGGIV,TÅB-V,...,None,ÖREVLIS,None,NEGGIV,ÖXAV,None,NEGGIV,ÖREBIV,RÄKSROTS,None
2,TURNUMMER,0303,0300,0309,1401,None,0320,None,0309,1405,...,None,0320,None,0309,1425,None,0315,1225,0925,None
3,ANMÄRKNING,,,,,None,,None,,,...,None,,None,,,None,,,,None
4,None,,,,07.45,None,,None,,13.00,...,None,,None,,13.00,None,,,16.45,None
5,None,,,,,,,None,,,...,,,None,,,,,,,
6,None,,,,08.00,None,,None,,13.17,...,None,,None,,13.17,None,,16.47,,
7,None,,,,08.10,None,08.15z,None,,13.27,...,None,08.15z,None,,13.27,None,,16.57,17.15,None
8,None,,,,,None,08.10z,None,,13.32,...,None,08.10z,None,,13.32,None,,17.02,17.20,None
9,None,,,,,None,,,,14.00,...,None,,,,14.00,None,,17.30,17.55,None


In [13]:
vessels = df.iloc[1]
turns   = df.iloc[2]
notes   = df.iloc[3]

meta = pd.DataFrame({
    "column": range(len(df.columns)),
    "vessel": vessels,
    "turn": turns,
    "note": notes
})

meta

,column,vessel,turn,note
0,0,FARTYG,TURNUMMER,ANMÄRKNING
1,1,NERAXAS,0303,
2,2,ÖREVLIS,0300,
3,3,NEGGIV,0309,
4,4,TÅB-V,1401,
5,5,None,None,None
6,6,ÖREVLIS,0320,
7,7,None,None,None
8,8,NEGGIV,0309,
9,9,TÅB-V,1405,


In [14]:
page = p.pages[0]

words = page.extract_words()

for w in words:
    print(w["text"])

GÄLLER
19
JUNI
2026
–
16
AUGUSTI
2026
Uppdaterad
24
juni
2026
3A
STOCKHOLM
–
VAXHOLM
–
NORRA
LAGNÖ
–
ÅLSTÄKET
DAG
MÅNDAG–TORSDAG
FREDAG
FARTYG
TURNUMMER
ANMÄRKNING
Strömkajen
(Stockholm)
Slussen
(Stockholm)
Nacka
strand
Hasseludden
Gåshaga
brygga
Vaxholm
ank.
Vaxholm
avg.
Grenadjärbryggan
(Rindö)
Knutsholmen
Skogsö
södra
Hästholmen
Skogsö
udde
Norra
Lagnö
Ålstäket
HELGTRAFIK
ANSLUTANDE
SL-TRAFIK
Vid
jul,
nyår,
påsk,
midsommar
samt
övriga
storhelger
Sök
hela
din
resa
på
sl.se
för
att
få
tider
för
både
förekommer
förändringar
i
trafiken.
Se
information
anslutande
SL-trafik
och
skärgårdstrafiken.
och
sök
din
resa
på
waxholmsbolaget.se
eller
i
appen.
NERAXAS
0303
06.28
06.45
ÖREVLIS
0300
06.45
07.00b
X
X
07.10
07.20
NEGGIV
0309
08.10
08.32
08.37b
X
X
08.47
08.55
09.10
TÅB-V
1401
07.45
08.00
08.10
ÖREVLIS
0320
08.15z
08.10z
08.50
NEGGIV
0309
09.32z
09.29z
09.26z
09.25z
08.55z
09.10
TÅB-V
1405
13.00
13.17
13.27
13.32
14.00
NEGGIV
0315
14.10
14.30
14.35b
X
X
14.45
14.55
15.10
TÅB-V
1305
17.00

# Ny test 


In [15]:
import requests
import pandas as pd

In [16]:
PIERS_URL = "https://map.stockholmarchipelagotrail.com/data/piers-identity.json"
VESSELS_URL = "https://map.stockholmarchipelagotrail.com/data/vessels-identity.json"

piers = requests.get(PIERS_URL).json()
vessels = requests.get(VESSELS_URL).json()

In [17]:
print(type(piers))
print(type(vessels))

print(len(piers))
print(len(vessels))

<class 'dict'>
<class 'dict'>
66
18


In [20]:
#list(piers.keys())  
print(type(piers))

first_key = next(iter(piers))

print(first_key)
print(piers[first_key])

<class 'dict'>
24hkr
{'uid': '24hkr', 'satId': 'sat:pier:24hkr', 'slug': 'alo', 'name': 'Ålö', 'concordances': {'gtfs': ['gtfs:9022001000224001'], 'slug': ['slug:alo']}, 'status': 'current', 'firstSeen': '2026-06-10'}


In [25]:
from unidecode import unidecode

def normalize(name):
    return (
        unidecode(name)
        .upper()
        .replace("-", " ")
        .replace("(", "")
        .replace(")", "")
        .strip()
    )

pier_lookup = {}

for pier in piers.values():
    key = normalize(pier["name"])
    pier_lookup[key] = pier
    print(key,pier)
#print(len(pier_lookup))

ALO {'uid': '24hkr', 'satId': 'sat:pier:24hkr', 'slug': 'alo', 'name': 'Ålö', 'concordances': {'gtfs': ['gtfs:9022001000224001'], 'slug': ['slug:alo']}, 'status': 'current', 'firstSeen': '2026-06-10'}
SANDHAMN {'uid': '25njf', 'satId': 'sat:pier:25njf', 'slug': 'sandhamn', 'name': 'Sandhamn', 'concordances': {'gtfs': ['gtfs:9022001000125001'], 'slug': ['slug:sandhamn']}, 'status': 'current', 'firstSeen': '2026-06-10'}
STROMKAJEN {'uid': '25rqy', 'satId': 'sat:pier:25rqy', 'slug': 'stromkajen', 'name': 'Strömkajen', 'concordances': {'gtfs': ['gtfs:9022001000301001', 'gtfs:9022001000301002', 'gtfs:9022001000301003', 'gtfs:9022001000301004', 'gtfs:9022001000301005', 'gtfs:9022001000301006', 'gtfs:9022001000301007', 'gtfs:9022001000301009', 'gtfs:9022001000301010', 'gtfs:9022001000301012', 'gtfs:9022001000301014', 'gtfs:9022001000301017'], 'slug': ['slug:stromkajen']}, 'status': 'current', 'firstSeen': '2026-06-10'}
LILLSVED {'uid': '2d2nq', 'satId': 'sat:pier:2d2nq', 'slug': 'lillsved', '

In [26]:
gtfs_to_sat = {}

for pier in piers.values():
    for gtfs in pier.get("concordances", {}).get("gtfs", []):
        gtfs_to_sat[gtfs] = pier["satId"]

print(len(gtfs_to_sat))

84


In [28]:
import re
from unidecode import unidecode

def normalize(text):
    if text is None:
        return None

    text = unidecode(str(text))
    text = text.upper()
    text = text.replace("(", "")
    text = text.replace(")", "")
    text = text.replace("-", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


TIME_RE = re.compile(r"^(\d\d)\.(\d\d)([abpz]?)$")


def parse_time(value):

    if value is None:
        return None

    value = value.strip()

    if value == "":
        return None

    if value == "X":
        return {
            "flexStop": True
        }

    m = TIME_RE.match(value)

    if not m:
        return {
            "raw": value
        }

    hh, mm, flag = m.groups()

    return {
        "time": f"{hh}:{mm}",
        "bookingRequired": flag == "b",
        "dropoffOnly": flag == "a",
        "pickupOnly": flag == "p",
        "reverseOrder": flag == "z"
    }

In [29]:
tests = [
    "07.00b",
    "08.53a",
    "17.03p",
    "09.32z",
    "14.10",
    "X"
]

for t in tests:
    print(t, "->", parse_time(t))

07.00b -> {'time': '07:00', 'bookingRequired': True, 'dropoffOnly': False, 'pickupOnly': False, 'reverseOrder': False}
08.53a -> {'time': '08:53', 'bookingRequired': False, 'dropoffOnly': True, 'pickupOnly': False, 'reverseOrder': False}
17.03p -> {'time': '17:03', 'bookingRequired': False, 'dropoffOnly': False, 'pickupOnly': True, 'reverseOrder': False}
09.32z -> {'time': '09:32', 'bookingRequired': False, 'dropoffOnly': False, 'pickupOnly': False, 'reverseOrder': True}
14.10 -> {'time': '14:10', 'bookingRequired': False, 'dropoffOnly': False, 'pickupOnly': False, 'reverseOrder': False}
X -> {'flexStop': True}


In [30]:
vessels = df.iloc[1]
turns = df.iloc[2]

trips = []

for col in range(len(df.columns)):

    vessel = vessels[col]
    trip = turns[col]

    if vessel is None:
        continue

    vessel = vessel.strip()

    if vessel == "":
        continue

    trips.append({
        "column": col,
        "vessel": vessel,
        "trip": trip
    })

len(trips)

21

In [31]:
pd.DataFrame(trips)

,column,vessel,trip
0,0,FARTYG,TURNUMMER
1,1,NERAXAS,0303
2,2,ÖREVLIS,0300
3,3,NEGGIV,0309
4,4,TÅB-V,1401
5,6,ÖREVLIS,0320
6,8,NEGGIV,0309
7,9,TÅB-V,1405
8,11,NEGGIV,0315
9,12,TÅB-V,1305


In [32]:
ROUTES = {

    "3A": [
        "Strömkajen",
        "Slussen",
        "Nacka strand",
        "Hasseludden",
        "Gåshaga brygga",
        "Vaxholm ank.",
        "Vaxholm avg.",
        "Grenadjärbryggan",
        "Knutsholmen",
        "Skogsö södra",
        "Hästholmen",
        "Skogsö udde",
        "Norra Lagnö",
        "Ålstäket"
    ]
}

# Ny start